In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import mne

DATA_ROOT = Path(r"C:\Users\Sanjeev\OneDrive\Desktop\Brain ID\EEG\ds004148")

OUTPUT_ROOT = DATA_ROOT / "processed_windows"
OUTPUT_ROOT.mkdir(exist_ok=True)

TARGET_CHANNELS = ["Fp1", "Fp2", "Fz"]

LOW_FREQ = 0.3
HIGH_FREQ = 45.0

WINDOW_SECONDS = 4
AMP_THRESHOLD_UV = 100.0

In [2]:
def preprocess_recording(vhdr_file):
    
    # -------------------------
    # Load BrainVision recording
    # -------------------------
    raw = mne.io.read_raw_brainvision(
        vhdr_file,
        preload=True,
        verbose=False
    )

    # -------------------------
    # Check channels
    # -------------------------
    missing = [
        ch for ch in TARGET_CHANNELS
        if ch not in raw.ch_names
    ]

    if missing:
        raise ValueError(
            f"Missing channels {missing} in {vhdr_file}"
        )

    # -------------------------
    # Select 3 channels
    # -------------------------
    raw_3ch = raw.copy().pick(TARGET_CHANNELS)

    # -------------------------
    # Bandpass filter
    # -------------------------
    filtered = raw_3ch.copy()

    filtered.filter(
        l_freq=LOW_FREQ,
        h_freq=HIGH_FREQ,
        fir_design="firwin",
        verbose=False
    )

    # -------------------------
    # Convert to numpy
    # -------------------------
    data = filtered.get_data()

    sfreq = filtered.info["sfreq"]

    # -------------------------
    # 4-second windows
    # -------------------------
    window_samples = int(
        WINDOW_SECONDS * sfreq
    )

    n_channels, n_samples = data.shape

    n_windows = n_samples // window_samples

    usable_samples = (
        n_windows * window_samples
    )

    data = data[:, :usable_samples]

    windows = data.reshape(
        n_channels,
        n_windows,
        window_samples
    )

    windows = np.transpose(
        windows,
        (1, 0, 2)
    )

    # Shape:
    # (windows, channels, samples)

    # -------------------------
    # Window QC
    # -------------------------
    qc_rows = []

    for idx, window in enumerate(windows):

        window_uv = window * 1e6

        has_nan_inf = not np.isfinite(
            window_uv
        ).all()

        ptp = np.ptp(
            window_uv,
            axis=1
        )

        std = np.std(
            window_uv,
            axis=1
        )

        frontal_artifact = (
            ptp[0] > AMP_THRESHOLD_UV or
            ptp[1] > AMP_THRESHOLD_UV
        )

        flatline = np.any(
            std < 0.1
        )

        reject = (
            has_nan_inf or
            frontal_artifact or
            flatline
        )

        qc_rows.append({
            "window": idx,

            "Fp1_ptp_uv": ptp[0],
            "Fp2_ptp_uv": ptp[1],
            "Fz_ptp_uv": ptp[2],

            "Fp1_std_uv": std[0],
            "Fp2_std_uv": std[1],
            "Fz_std_uv": std[2],

            "has_nan_inf": has_nan_inf,
            "frontal_artifact": frontal_artifact,
            "flatline": flatline,
            "reject": reject
        })

    qc_df = pd.DataFrame(qc_rows)

    # -------------------------
    # Keep only clean windows
    # -------------------------
    keep_mask = ~qc_df["reject"].to_numpy()

    clean_windows = windows[keep_mask]

    # Convert to float32 to reduce storage
    clean_windows = clean_windows.astype(
        np.float32
    )

    return (
        clean_windows,
        qc_df,
        sfreq
    )

In [3]:
vhdr_files = sorted(
    DATA_ROOT.glob(
        "sub-*/ses-*/eeg/*task-memory_eeg.vhdr"
    )
)

print("Memory recordings found:", len(vhdr_files))

Memory recordings found: 180


In [21]:
all_summary = []

for i, vhdr_file in enumerate(vhdr_files, start=1):

    subject = vhdr_file.parts[-4]
    session = vhdr_file.parts[-3]

    print(
        f"[{i:03d}/180] "
        f"{subject} | {session}"
    )

    try:

        clean_windows, qc_df, sfreq = (
            preprocess_recording(vhdr_file)
        )

        output_file = (
            OUTPUT_ROOT /
            f"{subject}_{session}.npz"
        )

        np.savez_compressed(
            output_file,
            windows=clean_windows,
            sfreq=sfreq
        )

        total_windows = len(qc_df)
        rejected_windows = int(qc_df["reject"].sum())
        accepted_windows = total_windows - rejected_windows

        rejection_rate = (
            rejected_windows / total_windows
        )

        all_summary.append({
            "subject": subject,
            "session": session,
            "total_windows": total_windows,
            "accepted_windows": accepted_windows,
            "rejected_windows": rejected_windows,
            "rejection_rate": rejection_rate,
            "output_file": str(output_file)
        })

    except Exception as e:

        print(
            f"ERROR: {subject} {session}: {e}"
        )

        all_summary.append({
            "subject": subject,
            "session": session,
            "total_windows": np.nan,
            "accepted_windows": np.nan,
            "rejected_windows": np.nan,
            "rejection_rate": np.nan,
            "output_file": None,
            "error": str(e)
        })

[001/180] sub-01 | ses-session1


[002/180] sub-01 | ses-session2
[003/180] sub-01 | ses-session3
[004/180] sub-02 | ses-session1
[005/180] sub-02 | ses-session2
[006/180] sub-02 | ses-session3
[007/180] sub-03 | ses-session1
[008/180] sub-03 | ses-session2
[009/180] sub-03 | ses-session3
[010/180] sub-04 | ses-session1
[011/180] sub-04 | ses-session2
[012/180] sub-04 | ses-session3
[013/180] sub-05 | ses-session1
[014/180] sub-05 | ses-session2
[015/180] sub-05 | ses-session3
[016/180] sub-06 | ses-session1
[017/180] sub-06 | ses-session2
[018/180] sub-06 | ses-session3
[019/180] sub-07 | ses-session1
[020/180] sub-07 | ses-session2
[021/180] sub-07 | ses-session3
[022/180] sub-08 | ses-session1
[023/180] sub-08 | ses-session2
[024/180] sub-08 | ses-session3
[025/180] sub-09 | ses-session1
[026/180] sub-09 | ses-session2
[027/180] sub-09 | ses-session3
[028/180] sub-10 | ses-session1
[029/180] sub-10 | ses-session2
[030/180] sub-10 | ses-session3
[031/180] sub-11 | ses-session1
[032/180] sub-11 | ses-session2
[033/180

In [ ]:
manifest = pd.DataFrame(all_summary)

manifest.to_csv(
    OUTPUT_ROOT / "preprocessing_manifest.csv",
    index=False
)

manifest.head()

,subject,session,total_windows,accepted_windows,rejected_windows,rejection_rate,output_file
0,sub-01,ses-session1,75,69,6,0.080000,C:\Users\Sanjeev\OneDrive\Desktop\Brain ID\EEG...
1,sub-01,ses-session2,75,73,2,0.026667,C:\Users\Sanjeev\OneDrive\Desktop\Brain ID\EEG...
2,sub-01,ses-session3,75,56,19,0.253333,C:\Users\Sanjeev\OneDrive\Desktop\Brain ID\EEG...
3,sub-02,ses-session1,75,33,42,0.560000,C:\Users\Sanjeev\OneDrive\Desktop\Brain ID\EEG...
4,sub-02,ses-session2,75,41,34,0.453333,C:\Users\Sanjeev\OneDrive\Desktop\Brain ID\EEG...


In [18]:
test_file = vhdr_files[0]

print(test_file)
print("Subject:", test_file.parts[-4])
print("Session:", test_file.parts[-3])

C:\Users\Sanjeev\OneDrive\Desktop\Brain ID\EEG\ds004148\sub-01\ses-session1\eeg\sub-01_ses-session1_task-memory_eeg.vhdr
Subject: sub-01
Session: ses-session1


In [6]:
print(
    "Successfully processed:",
    manifest["output_file"].notna().sum()
)

print(
    "Total recordings:",
    len(manifest)
)

Successfully processed: 180
Total recordings: 180


In [7]:
print(
    manifest[
        [
            "total_windows",
            "accepted_windows",
            "rejected_windows",
            "rejection_rate"
        ]
    ].describe()
)

       total_windows  accepted_windows  rejected_windows  rejection_rate
count     180.000000        180.000000        180.000000      180.000000
mean       74.977778         48.527778         26.450000        0.352913
std         0.298142         21.362639         21.326479        0.284750
min        71.000000          0.000000          1.000000        0.013333
25%        75.000000         36.000000          7.000000        0.093333
50%        75.000000         53.000000         22.000000        0.293333
75%        75.000000         68.000000         39.000000        0.520000
max        75.000000         74.000000         75.000000        1.000000


In [8]:
print(
    "Total windows:",
    int(manifest["total_windows"].sum())
)

print(
    "Total accepted:",
    int(manifest["accepted_windows"].sum())
)

print(
    "Total rejected:",
    int(manifest["rejected_windows"].sum())
)

print(
    "Overall rejection rate:",
    round(
        manifest["rejected_windows"].sum()
        /
        manifest["total_windows"].sum()
        * 100,
        2
    ),
    "%"
)

Total windows: 13496
Total accepted: 8735
Total rejected: 4761
Overall rejection rate: 35.28 %


In [9]:
print("Successfully processed:", manifest["output_file"].notna().sum())
print("Total recordings:", len(manifest))

print(
    manifest[
        [
            "total_windows",
            "accepted_windows",
            "rejected_windows",
            "rejection_rate"
        ]
    ].describe()
)

print(
    "Overall rejection rate:",
    round(
        manifest["rejected_windows"].sum()
        /
        manifest["total_windows"].sum()
        * 100,
        2
    ),
    "%"
)

Successfully processed: 180
Total recordings: 180
       total_windows  accepted_windows  rejected_windows  rejection_rate
count     180.000000        180.000000        180.000000      180.000000
mean       74.977778         48.527778         26.450000        0.352913
std         0.298142         21.362639         21.326479        0.284750
min        71.000000          0.000000          1.000000        0.013333
25%        75.000000         36.000000          7.000000        0.093333
50%        75.000000         53.000000         22.000000        0.293333
75%        75.000000         68.000000         39.000000        0.520000
max        75.000000         74.000000         75.000000        1.000000
Overall rejection rate: 35.28 %


In [11]:
# Recordings with >80% rejection
high_rejection = manifest[
    manifest["rejection_rate"] > 0.80
]

print(
    "Recordings with >80% rejection:",
    len(high_rejection)
)

print(
    high_rejection[
        [
            "subject",
            "session",
            "accepted_windows",
            "rejected_windows",
            "rejection_rate"
        ]
    ]
)

Recordings with >80% rejection: 19
          subject session  accepted_windows  rejected_windows  rejection_rate
5    ses-session3     eeg                 1                74        0.986667
7    ses-session2     eeg                11                64        0.853333
13   ses-session2     eeg                12                63        0.840000
30   ses-session1     eeg                 1                74        0.986667
31   ses-session2     eeg                 3                72        0.960000
37   ses-session2     eeg                 8                67        0.893333
38   ses-session3     eeg                 0                75        1.000000
83   ses-session3     eeg                12                59        0.830986
93   ses-session1     eeg                 9                66        0.880000
98   ses-session3     eeg                 0                75        1.000000
128  ses-session3     eeg                 8                67        0.893333
131  ses-session3     eeg    

In [25]:
session_summary = (
    manifest
    .groupby("session")
    .agg(
        recordings=("subject", "count"),
        total_windows=("total_windows", "sum"),
        accepted_windows=("accepted_windows", "sum"),
        rejected_windows=("rejected_windows", "sum")
    )
)

session_summary["rejection_rate"] = (
    session_summary["rejected_windows"] /
    session_summary["total_windows"]
)

session_summary

,recordings,total_windows,accepted_windows,rejected_windows,rejection_rate
session,,,,,
ses-session1,60,4500,3060,1440,0.320000
ses-session2,60,4500,3040,1460,0.324444
ses-session3,60,4496,2635,1861,0.413923


For deleting Existence Preprocessed data 

In [20]:
import shutil

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

OUTPUT_ROOT.mkdir(exist_ok=True)

print("Old processed output removed.")

Old processed output removed.
